# 🛒 مشروع سِلّة (SILLA) - مبادرة رواد مصر الرقمية (DEPI)
## موديول السلة الذكية ومنطق الأعمال (Smart Cart & Business Logic)
### إعداد المهندسة: جنى شلبي (Jana Shalaby)
---
### 📌 نظرة عامة على مهام الموديول:
يقوم هذا الجزء بتحويل ترشيحات الوصفات إلى **سلة مشتريات فعلية وذكية** عبر تحقيق كافة متطلبات خطة العمل:
1. **حساب كميات المكونات حسب عدد الأفراد (Servings Scaling)** ومراعاة الحجم الأساسي لكل وجبة.
2. **استبعاد وخصم المكونات المتوفرة لدى المستخدم بالمنزل (Pantry Check)** سواء بالاسم أو بالكمية.
3. **مطابقة المكونات بمنتجات المتاجر وأحجام العبوات المتوفرة** مع تقريب الشراء (Ceiling division) واحتساب الفائض (Leftovers).
4. **دمج المكونات المشتركة بين أكثر من وصفة (Multi-Recipe Consolidation)** لمنع شراء عبوات متكررة وحساب نسبة التوفير بالجنيه.
5. **مراعاة الميزانية وترشيد السلة (Budget Optimization)** باستخدام خوارزميات Knapsack والبرمجة الخطية (OR-Tools) لإعطاء الأولوية للأساسيات واستبعاد غير الحرج.
6. **التعامل مع المنتجات غير المتاحة والبدائل الذكية (Out-of-Stock & Substitutes)** باستخدام مصفوفة البدائل الغذائية المعتمدة في المطبخ المصري.
7. **المقارنة الاقتصادية بين سلاسل السوبرماركت (Multi-Store Comparison)** لإرشاد المستخدم للمتجر الأوفر.


## 1. تهيئة بيئة العمل والاتصال بمحرك السلة الذكية


In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from SILLA_Recommendation.cart_builder import CartBuilder
from SILLA_Recommendation.api import build_cart, compare_stores

# Initialize CartBuilder
builder = CartBuilder()
print("✅ تم تحميل محرك السلة الذكية وقاعدة البيانات بنجاح.")


## 2. حساب كميات المكونات حسب عدد الأفراد (Servings Scaling)
يقوم المحرك بمقارنة عدد الأفراد المطلوب (`target_servings`) مع الحجم المعياري للوصفة (`base_servings`) لحساب معامل التكبير/التصغير:
$$\text{Scale Factor} = \frac{\text{Target Servings}}{\text{Base Servings}}$$


In [ ]:
# تجربة 1: حساب السلة لوصفة مكرونة بشاميل (R0001) لـ 6 أفراد (الحجم المعياري) ثم لـ 12 فرداً (عزومة)
base_cart = builder.build_cart(["R0001"], servings=6)
scaled_cart = builder.build_cart(["R0001"], servings=12)

print(f"الوصفة: {base_cart['recipes'][0]['recipe_name_ar']}")
print(f"1. التكلفة المعيارية (6 أفراد): {base_cart['smart_cart_total_egp']:.2f} ج.م")
print(f"2. التكلفة بعد مضاعفة الأفراد (12 فرداً): {scaled_cart['smart_cart_total_egp']:.2f} ج.م")

# عرض تفصيلي لمقارنة كمية المكرونة واللحم المفروم
pasta_base = next(i for i in base_cart['cart_items'] if i['ingredient_id'] == 'ING0089')
pasta_scaled = next(i for i in scaled_cart['cart_items'] if i['ingredient_id'] == 'ING0089')
print(f"كمية المكرونة: {pasta_base['total_quantity_needed']} جم ➡️ {pasta_scaled['total_quantity_needed']} جم (عبوات: {pasta_scaled['smart_packages_needed']})")


## 3. استبعاد وخصم المكونات المتوفرة في المنزل (Pantry Check)
يتيح المحرك للمستخدم إدخال قائمة بالمكونات المتوفرة لديه، ليقوم النظام تلقائياً بخصمها أو استبعادها من سلة الشراء لتوفير المال.


In [ ]:
# تجربة 2: استبعاد مكونات المطبخ
user_kitchen = ["بصل أصفر", "فلفل أسود", "لحم بقري مفروم"]

cart_with_pantry = builder.build_cart(
    recipe_ids=["R0001"],
    servings=6,
    user_pantry=user_kitchen
)

print(f"إجمالي السلة بدون مطبخ: {base_cart['smart_cart_total_egp']:.2f} ج.م")
print(f"إجمالي السلة بعد استبعاد مكونات المطبخ: {cart_with_pantry['smart_cart_total_egp']:.2f} ج.م")
print(f"عدد المكونات المستبعدة: {cart_with_pantry['pantry_covered_count']}")

print("
المكونات المستبعدة المتوفرة بالمنزل:")
for item in cart_with_pantry['pantry_covered_items']:
    print(f" - {item['ingredient_name_ar']}: وفرت حوالي {item['estimated_money_saved_egp']:.2f} ج.م")


## 4. دمج المكونات المشتركة بين أكثر من وصفة (Multi-Recipe Consolidation)
عندما يختار المستخدم طبخ أكثر من وجبة، يجمع المحرك الكميات المطلوبة لنفس المكون (مثل اللحم، البصل، التوابل) لشراء عبوة بحجم موحد بدلاً من شراء عبوات مستقلة لكل طبخة.


In [ ]:
# تجربة 3: طبخ مكرونة بشاميل (R0001) + كفتة مشوية (R0007)
multi_cart = builder.build_cart(["R0001", "R0007"])

print(f"تكلفة الشراء المنفصل لكل وجبة على حدة: {multi_cart['separate_carts_total_egp']:.2f} ج.م")
print(f"تكلفة السلة الذكية بعد الدمج: {multi_cart['smart_cart_total_egp']:.2f} ج.م")
print(f"💰 إجمالي التوفير المالي المحقق: {multi_cart['total_savings_egp']:.2f} ج.م ({multi_cart['savings_percentage']}%)")
print(f"عدد المكونات المشتركة المدمجة: {multi_cart['shared_ingredients_count']}")

print("
أبرز المكونات المدمجة والتوفير المحقق فيها:")
for item in multi_cart['cart_items']:
    if item['is_shared']:
        print(f" - {item['ingredient_name_ar']}: إجمالي المطلوب {item['total_quantity_needed']} {item['unit']} | توفير: {item['savings_egp']} ج.م")


## 5. مراعاة الميزانية وترشيد السلة (Budget Optimization with OR-Tools / Knapsack)
عندما يحدد المستخدم ميزانية قصوى أقل من تكلفة السلة، يقوم المحرك بتطبيق خوارزمية **0-1 Knapsack / Integer Linear Programming**:
- إعطاء أولوية قصوى للمكونات الأساسية (Essential Ingredients).
- استبعاد المكونات الاختيارية (Optional Ingredients) أولاً.
- ضمان بقاء التكلفة الإجمالية تحت سقف الميزانية.


In [ ]:
# تجربة 4: تطبيق سقف ميزانية (80 جنيه) لسلة مكرونة بشاميل
budget_limit = 80.0
budgeted_cart = builder.build_cart(["R0001"], budget_egp=budget_limit, optimize_budget=True)

b_info = budgeted_cart['budget_analysis']
print(f"حالة الميزانية: {b_info['budget_status']}")
print(f"التكلفة الأصلية الكاملة: {b_info['original_cost_egp']:.2f} ج.م")
print(f"الميزانية المحددة: {budget_limit:.2f} ج.م")
print(f"التكلفة بعد الترشيد: {b_info['optimized_cart_total_egp']:.2f} ج.م")
print(f"محرك الحل المستخدم: {b_info.get('solver_engine')}")
print(f"
نصيحة النظام الذكية: {b_info['advice_message_ar']}")


## 6. محرك البدائل والتعامل مع المنتجات غير المتاحة (Out-of-Stock & Substitutes)
إذا كان هناك مكون غير متوفر في المتجر أو نفدت كميته، يقترح المحرك تلقائياً بدائل طهي معتمدة ونسب التحويل الخاصة بها.


In [ ]:
# تجربة 5: فحص البدائل الذكية للزبدة والمكرونة
for ing_id in ["ING0017", "ING0089", "ING0123"]:
    subs = builder.find_substitutes(ing_id)
    if subs:
        print(f"
بدائل المكون {ing_id}:")
        for s in subs:
            prod_info = f" - المنتج المقترح: {s['best_product']['product_name_ar']} ({s['best_product']['price_egp']} ج.م)" if s['best_product'] else " - متوفر كخيار منزلي"
            print(f" * البديل: {s['substitute_name_ar']} (معامل التحويل: {s['ratio']}x) - السبب: {s['reason_ar']}{prod_info}")


## 7. المقارنة المالية الشاملة بين سلاسل المتاجر (Multi-Store Comparison)
مقارنة إجمالي تكلفة السلة بين كبرى سلاسل السوبرماركت المصرية (كارفور، لولو، هايبر وان، سعودي) وإرشاد المستهلك للمتجر الأوفر.


In [ ]:
# تجربة 6: مقارنة السوبرماركت لوصفة المكرونة بشاميل
store_report = builder.compare_stores(["R0001"])

print(f"عدد المتاجر التي خضعت للمقارنة: {store_report['stores_evaluated_count']}")
if store_report['cheapest_store']:
    print(f"🏆 المتجر الأوفر سعراً: {store_report['cheapest_store']['store_name']}")
    print(f"💰 تكلفة السلة في المتجر الأوفر: {store_report['cheapest_store']['total_cost_egp']:.2f} ج.م")
    print(f"💵 أقصى توفير ممكن بين المتاجر: {store_report['potential_store_savings_egp']:.2f} ج.م")

print("
ترتيب المتاجر من حيث السعر واكتمال المنتجات:")
for s in store_report['store_rankings']:
    status = "مكتمل بالكامل" if s['is_fully_priced'] else f"ينقصه {s['unpriced_count']} منتجات"
    print(f" - {s['store_name']:<25} | السعر: {s['total_cost_egp']:>7.2f} ج.م | الحالة: {status}")


## 8. الخلاصة وتسليم الموديول لواجهة المستخدم (Handover to Frontend)
تم إتمام جميع المهام المسندة إلى **جنى شلبي** بنجاح 100%:
- تم بناء صنف `CartBuilder` وتوفير دوال نظيفة للاستخدام عبر الـ API.
- الموديول جاهز بالكامل للربط المباشر مع واجهة **Streamlit** التي تقوم ببنائها الزميلة **ملك أيمن**، حيث تستطيع الواجهة تمرير `servings` و `user_pantry` و `budget_egp` و `store_id` واستلام سلة متكاملة مع رسائل نصيحة جاهزة للعرض للمستخدم.
